# Benchmark: BorutaPy vs Bankai Boruta

Este notebook compara tempo de ajuste e qualidade da seleção em dados sintéticos com atributos relevantes conhecidos. Ele apresenta duas leituras:

1. **Uso prático:** BorutaPy com `RandomForestClassifier` do scikit-learn versus Bankai com `BankaiRandomForestClassifier`, `max_bins=63` e `n_jobs=4`. O baseline também usa `n_jobs=4`.
2. **Controller controlado:** BorutaPy e Bankai usam a mesma configuração de `RandomForestClassifier`, também com `n_jobs=4`.

Os resultados dependem da máquina, das versões instaladas e da carga do sistema. Execute todas as repetições no mesmo ambiente. O tempo inclui apenas `fit`; criação dos dados, imports e construção dos estimadores ficam fora da medição.

Para recriar as dependências no ambiente: `.venv/bin/python -m pip install 'bankai-boruta[bankai-rf]==0.1.0a2' 'Boruta==0.4.3' jupyter ipykernel matplotlib pandas`.

Abra com o ambiente criado para este benchmark: `.venv/bin/jupyter lab notebooks/benchmark_borutapy_vs_bankai.ipynb`.

In [ ]:
import importlib.metadata as metadata
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from boruta import BorutaPy
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

from bankai_boruta import BankaiBoruta

print({
    'bankai-boruta': metadata.version('bankai-boruta'),
    'Boruta': metadata.version('Boruta'),
    'scikit-learn': sklearn.__version__,
    'numpy': np.__version__,
})

## Protocolo

O benchmark varre o número de linhas e mantém fixos atributos, árvores, iterações e seeds pareadas. `shuffle=False` mantém as colunas informativas nas primeiras posições, permitindo medir precision e recall da seleção. Todas as florestas usam quatro workers; as iterações Boruta continuam sequenciais.

O notebook passa `max_bins=63` explicitamente ao Bankai Random Forest. Na versão publicada instalada (`0.1.0a10`), o valor padrão da assinatura é `None`; portanto, 63 bins não é o default desse release.

In [ ]:
SAMPLE_SIZES = (2_000, 5_000, 10_000, 20_000)
N_FEATURES = 32
N_INFORMATIVE = 5
N_TREES = 48
MAX_ITER = 20
N_REPEATS = 2
DATA_SEED = 2026
ALPHA = 0.01
N_JOBS = 4
MAX_BINS = 63

known_relevant = np.arange(N_INFORMATIVE)
print(f'tamanhos={SAMPLE_SIZES}, atributos={N_FEATURES}, relevantes={known_relevant.tolist()}')

## Métodos comparados

A comparação principal usa BorutaPy com RandomForest scikit-learn e Bankai com Bankai Random Forest histogram-based (`max_bins=63`, `n_jobs=4`, importance gain). A comparação controlada usa o mesmo RandomForest scikit-learn nos dois controllers. O modo controlado ajuda a isolar os controllers; diferenças estatísticas entre implementações Boruta ainda podem alterar as decisões.

In [ ]:
from bankai_random_forest import BankaiRandomForestClassifier

METHODS = ('BorutaPy + sklearn RF', 'Bankai + Bankai RF (63 bins)', 'Bankai + sklearn RF')

def make_sklearn_forest(seed):
    return RandomForestClassifier(
        n_estimators=N_TREES,
        max_depth=16,
        max_features='sqrt',
        class_weight='balanced',
        n_jobs=N_JOBS,
        random_state=seed,
    )

def make_selector(method, seed):
    if method == 'BorutaPy + sklearn RF':
        return BorutaPy(
            estimator=make_sklearn_forest(seed),
            n_estimators=N_TREES,
            max_iter=MAX_ITER,
            alpha=ALPHA,
            perc=100,
            two_step=False,
            random_state=seed,
            verbose=0,
        )
    if method == 'Bankai + Bankai RF (63 bins)':
        estimator = BankaiRandomForestClassifier(
            n_estimators=N_TREES,
            max_depth=16,
            max_features='sqrt',
            class_weight='balanced',
            n_jobs=N_JOBS,
            random_state=seed,
            importance_type='gain',
            max_bins=MAX_BINS,
        )
        return BankaiBoruta(
            estimator=estimator, importance_getter='auto',
            n_estimators=N_TREES, max_iter=MAX_ITER, alpha=ALPHA,
            max_depth=16, max_features='sqrt', n_jobs=N_JOBS, random_state=seed,
        )
    if method == 'Bankai + sklearn RF':
        return BankaiBoruta(
            estimator=make_sklearn_forest(seed),
            importance_getter='auto',
            n_estimators=N_TREES, max_iter=MAX_ITER, alpha=ALPHA,
            max_depth=16, max_features='sqrt', n_jobs=N_JOBS, random_state=seed,
        )
    raise ValueError(method)

## Executar benchmark

A ordem dos métodos é embaralhada em cada repetição para reduzir efeitos de aquecimento ou carga crescente. As métricas de seleção usam apenas atributos aceitos (`support_`); atributos tentativos ficam de fora e podem ser analisados separadamente.

In [ ]:
def selection_metrics(selected, relevant):
    selected = set(np.flatnonzero(selected).tolist())
    relevant = set(np.asarray(relevant).tolist())
    true_positive = len(selected & relevant)
    precision = true_positive / len(selected) if selected else 0.0
    recall = true_positive / len(relevant) if relevant else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

def make_dataset(n_samples):
    X, y = make_classification(
        n_samples=n_samples, n_features=N_FEATURES, n_informative=N_INFORMATIVE,
        n_redundant=0, n_repeated=0, n_classes=2, class_sep=1.0,
        shuffle=False, random_state=DATA_SEED,
    )
    return np.ascontiguousarray(X, dtype=np.float64), np.asarray(y)

rows = []
order_rng = random.Random(DATA_SEED)
for n_samples in SAMPLE_SIZES:
    X, y = make_dataset(n_samples)
    for repeat in range(N_REPEATS):
        seed = DATA_SEED + repeat
        order = list(METHODS)
        order_rng.shuffle(order)
        for method in order:
            selector = make_selector(method, seed)
            started = time.perf_counter()
            selector.fit(X, y)
            elapsed = time.perf_counter() - started
            selected = selector.support_
            precision, recall, f1 = selection_metrics(selected, known_relevant)
            n_iter = getattr(selector, 'n_iter_', None)
            if n_iter is None and hasattr(selector, 'importance_history_'):
                n_iter = selector.importance_history_.shape[0] - 1
            rows.append({
                'n_samples': n_samples, 'method': method, 'repeat': repeat + 1, 'seed': seed,
                'seconds': elapsed, 'n_selected': int(np.sum(selected)),
                'precision': precision, 'recall': recall, 'f1': f1,
                'n_iter': int(n_iter),
            })

results = pd.DataFrame(rows)
results.sort_values(['n_samples', 'repeat', 'method']).reset_index(drop=True)

In [ ]:
summary = (
    results.groupby(['n_samples', 'method'])
    .agg(
        median_seconds=('seconds', 'median'),
        mean_seconds=('seconds', 'mean'),
        std_seconds=('seconds', 'std'),
        median_selected=('n_selected', 'median'),
        median_precision=('precision', 'median'),
        median_recall=('recall', 'median'),
        median_f1=('f1', 'median'),
        median_iterations=('n_iter', 'median'),
    )
    .sort_index()
)
summary

## Resultados da rodada de 2026-10-03

Configuração fixa: 32 atributos, 5 informativos, 48 árvores, até 20 iterações, duas repetições, `n_jobs=4` e `max_bins=63` no Bankai RF. Execução em Python 3.11, Bankai Boruta 0.1.0a2, Bankai RF 0.1.0a10, Boruta 0.4.3, scikit-learn 1.9.1, NumPy 2.4.6, macOS ARM64.

| Linhas | BorutaPy + sklearn RF (s) | Bankai + Bankai RF (s) | Bankai + sklearn RF (s) | F1 dos métodos |
|---:|---:|---:|---:|---:|
| 2.000 | 1,987 | 0,596 | 2,325 | 1,00 |
| 5.000 | 4,549 | 1,385 | 5,070 | 1,00 |
| 10.000 | 9,946 | 2,410 | 10,222 | 1,00 |
| 20.000 | 23,365 | 4,904 | 24,507 | 1,00 |

Bankai + Bankai RF ficou aproximadamente 3,3×, 3,3×, 4,1× e 4,8× mais rápido que BorutaPy + sklearn RF nos tamanhos de 2k, 5k, 10k e 20k, respectivamente. Ao passar de 2k para 20k linhas (10× os dados), o tempo mediano cresceu 8,2× no Bankai histogram e 11,8× no baseline BorutaPy. Todos selecionaram os cinco atributos relevantes nas duas repetições em cada tamanho. São resultados sintéticos de uma máquina; aumente repetições e varie a dimensionalidade antes de generalizar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
times = summary['median_seconds'].unstack('method')
times.plot(marker='o', ax=axes[0])
axes[0].set_title('Tempo mediano de fit por número de linhas')
axes[0].set_xlabel('linhas')
axes[0].set_ylabel('segundos (menos é melhor)')
axes[0].set_xticks(SAMPLE_SIZES)
quality = summary['median_f1'].unstack('method')
quality.plot(marker='o', ax=axes[1])
axes[1].set_title('F1 da seleção')
axes[1].set_xlabel('linhas')
axes[1].set_ylabel('F1 (mais é melhor)')
axes[1].set_xticks(SAMPLE_SIZES)
plt.tight_layout()
plt.show()

## Interpretação

Compare medianas e dispersão, não apenas uma execução. O F1 mede recuperação dos atributos informativos desta simulação; não é uma medida universal de qualidade em dados reais. A comparação prática mede as configurações padrão de cada produto. A linha **Bankai + sklearn RF** é a comparação mais controlada para avaliar o controller Bankai frente ao BorutaPy.

Para tornar a medição mais estável, aumente `N_REPEATS`; para avaliar outras faixas de escala, altere `SAMPLE_SIZES` e `N_FEATURES`. Registre a máquina e as versões impressas acima ao compartilhar os resultados.